# Results -- HTR Error Dataset evaluation

Aggregates the raw per-system predictions written by `evaluate.py`
(`results/predictions/*.csv`) into headline metrics, figures, and summary
tables. Run `evaluate.py` for at least one system before running this
notebook -- see README.md.

Core measurements:
- **CER** (character error rate) on clean vs. corrupted images.
- **Faithful rate**: share of corrupted-image predictions that match the
  text actually displayed (the corrupted word) -- faithful visual
  grounding.
- **Repair rate**: share of corrupted-image predictions that instead match
  the *original*, uncorrupted word -- the system "fixed" an implausible
  stimulus using a linguistic prior instead of reading what is on the
  page.
- **Minimal-pair effect**: whether repair is more frequent when the
  corruption happens to land on another real word.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from plotting import apply_style, style_ax, save_fig, BLUE, ORANGE, AQUA, RED, VIOLET, GREEN, INK_MUTED
from textmetrics import levenshtein, normalize_prediction, load_english_wordset
from systems import SYSTEMS_BY_NAME, PROMPT_VARIANTS

apply_style()

DATASET_DIR = Path("dataset")
PRED_DIR = Path("results/predictions")
RESULTS_DIR = Path("results")
FIG_DIR = Path("results/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_VARIANT = "default"  # a system's own configured prompt (systems.py)
VARIANT_COLOR = {DEFAULT_VARIANT: BLUE, "diplomatic": ORANGE, "symbol_game": AQUA}
VARIANT_LABEL = {
    DEFAULT_VARIANT: "Own default prompt",
    "diplomatic": "Diplomatic-transcription prompt",
    "symbol_game": "Symbol-game prompt",
}


## 1. Load predictions and ground truth

In [ ]:
def parse_pred_filename(path: Path) -> tuple[str, str]:
    """'<system>.csv' -> (system, 'default'); '<system>__<variant>.csv' -> (system, variant)."""
    stem = path.stem
    if "__" in stem:
        system, variant = stem.split("__", 1)
        return system, variant
    return stem, DEFAULT_VARIANT


def load_predictions() -> pd.DataFrame:
    frames = []
    found_systems = set()
    for path in sorted(PRED_DIR.glob("*.csv")):
        system, variant = parse_pred_filename(path)
        if system not in SYSTEMS_BY_NAME:
            print(f"WARNING: {path} doesn't match a known system name -- skipped")
            continue
        df = pd.read_csv(path, dtype={"id": str})
        if df.empty:
            continue
        df["system"] = system
        df["variant"] = variant
        frames.append(df)
        found_systems.add(system)
    missing = sorted(set(SYSTEMS_BY_NAME) - found_systems)
    for name in missing:
        print(f"WARNING: no predictions for {name} -- skipped")
    if not frames:
        raise SystemExit("No prediction files found in results/predictions/. Run evaluate.py first.")
    return pd.concat(frames, ignore_index=True)


meta = pd.read_csv(DATASET_DIR / "metadata.csv", dtype={"id": str})
wordset = load_english_wordset()
meta["word_is_dict"] = meta["word"].str.lower().isin(wordset)
meta["error_is_dict"] = meta["error_word"].str.lower().isin(wordset)
meta["minimal_pair"] = meta["word_is_dict"] & meta["error_is_dict"]

raw_preds = load_predictions()
print(f"{len(raw_preds)} raw prediction rows, systems: {sorted(raw_preds['system'].unique())}")


## 2. Score every prediction

In [ ]:
def score(preds: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    preds = preds.copy()
    preds["pred_norm"] = preds["prediction"].apply(normalize_prediction)
    preds["gt_norm"] = preds["gt_text"].str.lower()
    preds = preds.merge(meta[["id", "word", "error_word", "minimal_pair"]], on="id", how="left")

    preds["cer"] = [
        levenshtein(p, g) / max(1, len(g)) for p, g in zip(preds["pred_norm"], preds["gt_norm"])
    ]
    preds["exact_match"] = preds["pred_norm"] == preds["gt_norm"]
    preds["repaired_to_original"] = False
    err_mask = preds["condition"] == "error"
    preds.loc[err_mask, "repaired_to_original"] = (
        preds.loc[err_mask, "pred_norm"] == preds.loc[err_mask, "word"].str.lower()
    )
    return preds


preds = score(raw_preds, meta)
preds.to_csv(RESULTS_DIR / "predictions_scored.csv", index=False)
preds.head()


## 3. Summarize per (system, prompt variant)

In [ ]:
def summarize(preds: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (system, variant), sub in preds.groupby(["system", "variant"]):
        spec = SYSTEMS_BY_NAME[system]
        clean = sub[sub["condition"] == "clean"]
        error = sub[sub["condition"] == "error"]
        mp_error = error[error["minimal_pair"]]
        nonmp_error = error[~error["minimal_pair"]]
        rows.append({
            "system": system,
            "variant": variant,
            "display_name": spec.display_name,
            "kind": spec.kind,
            "n_clean": len(clean),
            "n_error": len(error),
            "cer_clean": clean["cer"].mean(),
            "cer_error": error["cer"].mean(),
            "cer_paired_increase": error["cer"].mean() - clean["cer"].mean(),
            "word_acc_clean": clean["exact_match"].mean(),
            "faithful_rate_error": error["exact_match"].mean(),
            "repair_rate_error": error["repaired_to_original"].mean(),
            "faithful_rate_minimal_pair": mp_error["exact_match"].mean() if len(mp_error) else float("nan"),
            "repair_rate_minimal_pair": mp_error["repaired_to_original"].mean() if len(mp_error) else float("nan"),
            "faithful_rate_non_minimal_pair": nonmp_error["exact_match"].mean() if len(nonmp_error) else float("nan"),
            "mean_latency_sec": sub["elapsed_sec"].mean(),
        })
    return pd.DataFrame(rows)


all_summary = summarize(preds)
all_summary.to_csv(RESULTS_DIR / "metrics_summary_variants.csv", index=False)

summary = all_summary[all_summary["variant"] == DEFAULT_VARIANT].sort_values("cer_error")
summary.to_csv(RESULTS_DIR / "metrics_summary.csv", index=False)
summary


In [ ]:
variant_summary = all_summary[all_summary["variant"] != DEFAULT_VARIANT]
if not variant_summary.empty:
    print("--- prompt-variant runs ---")
    display(variant_summary.sort_values(["system", "variant"]))
else:
    print("No prompt-variant runs found (results/predictions/<system>__<variant>.csv).")


## 4. Figures

### 4.1 Character error rate: clean vs. corrupted


In [ ]:
s = summary.sort_values("cer_error")
x = np.arange(len(s))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - 0.2, s["cer_clean"] * 100, width=0.4, color=BLUE, label="Clean image")
ax.bar(x + 0.2, s["cer_error"] * 100, width=0.4, color=ORANGE, label="Corrupted image")
ax.set_xticks(x)
ax.set_xticklabels(s["display_name"], rotation=35, ha="right")
style_ax(ax)
ax.legend(frameon=False)
ax.set_ylabel("Character error rate (%)")
save_fig(fig, "results_fig1_cer_clean_vs_error", FIG_DIR)
plt.show()


### 4.2 Faithful vs. repair rate on corrupted images

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - 0.2, s["faithful_rate_error"] * 100, width=0.4, color=AQUA,
       label="Faithful (reads the displayed text)")
ax.bar(x + 0.2, s["repair_rate_error"] * 100, width=0.4, color=RED,
       label="Repaired (returns the original word instead)")
ax.set_xticks(x)
ax.set_xticklabels(s["display_name"], rotation=35, ha="right")
style_ax(ax)
ax.legend(frameon=False)
ax.set_ylabel("Share of corrupted-image predictions (%)")
save_fig(fig, "results_fig2_faithful_vs_repair", FIG_DIR)
plt.show()


### 4.3 Minimal-pair effect

Repair rate when the corruption happens to land on another real English
word (e.g. *said* -> *laid*) versus the faithful rate on corruptions that
are not real words. A higher repair rate on minimal pairs indicates
stronger linguistic-prior reliance.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - 0.2, s["repair_rate_minimal_pair"] * 100, width=0.4, color=VIOLET,
       label="Corruption also a real word (minimal pair)")
ax.bar(x + 0.2, s["faithful_rate_non_minimal_pair"] * 100, width=0.4, color=GREEN,
       label="Corruption is a non-word (faithful rate)")
ax.set_xticks(x)
ax.set_xticklabels(s["display_name"], rotation=35, ha="right")
style_ax(ax)
ax.legend(frameon=False)
ax.set_ylabel("Rate (%)")
save_fig(fig, "results_fig3_minimal_pair_effect", FIG_DIR)
plt.show()


### 4.4 Prompt-variant comparison

Only produced for systems that have at least one non-default prompt
variant recorded (i.e. `evaluate.py` was run with `--prompt-variant`).


In [ ]:
systems_with_variants = sorted(
    all_summary.loc[all_summary["variant"] != DEFAULT_VARIANT, "system"].unique()
)

if not systems_with_variants:
    print("No prompt-variant runs found -- skipping.")
else:
    variants_present = [DEFAULT_VARIANT] + [v for v in PROMPT_VARIANTS if
                                             (all_summary["variant"] == v).any()]
    vs = all_summary[all_summary["system"].isin(systems_with_variants)]
    display_names = [SYSTEMS_BY_NAME[sv].display_name for sv in systems_with_variants]
    xv = np.arange(len(systems_with_variants))
    width = 0.8 / len(variants_present)

    for metric, fname, ylabel in [
        ("faithful_rate_error", "results_fig4_prompt_variants_faithful", "Faithful rate on corrupted images (%)"),
        ("repair_rate_error", "results_fig5_prompt_variants_repair", "Repair rate on corrupted images (%)"),
    ]:
        fig, ax = plt.subplots(figsize=(9, 4))
        for i, variant in enumerate(variants_present):
            vals = []
            for sysname in systems_with_variants:
                row = vs[(vs["system"] == sysname) & (vs["variant"] == variant)]
                vals.append(row[metric].iloc[0] * 100 if len(row) else np.nan)
            offset = (i - (len(variants_present) - 1) / 2) * width
            ax.bar(xv + offset, vals, width=width, color=VARIANT_COLOR.get(variant, INK_MUTED),
                   label=VARIANT_LABEL.get(variant, variant))
        ax.set_xticks(xv)
        ax.set_xticklabels(display_names, rotation=35, ha="right")
        style_ax(ax)
        ax.legend(frameon=False)
        ax.set_ylabel(ylabel)
        save_fig(fig, fname, FIG_DIR)
        plt.show()


## 5. Headline summary

In [ ]:
def _fmt_pct(x):
    return f"{x*100:.1f}%" if pd.notna(x) else "--"


best_faithful = s.loc[s["faithful_rate_error"].idxmax()]
worst_faithful = s.loc[s["faithful_rate_error"].idxmin()]
highest_repair = s.loc[s["repair_rate_error"].idxmax()]

print(f"Most faithful on corrupted images:  {best_faithful['display_name']} "
      f"({_fmt_pct(best_faithful['faithful_rate_error'])} exact match)")
print(f"Least faithful on corrupted images: {worst_faithful['display_name']} "
      f"({_fmt_pct(worst_faithful['faithful_rate_error'])} exact match)")
print(f"Strongest repair behaviour:         {highest_repair['display_name']} "
      f"({_fmt_pct(highest_repair['repair_rate_error'])} of corrupted trials "
      f"returned the ORIGINAL word instead of the one actually shown)")

n_pairs = preds["id"].nunique()
total_dataset_n = len(meta)
if n_pairs < total_dataset_n:
    print(f"\nNote: this is a partial run ({n_pairs} of {total_dataset_n} dataset pairs). "
          f"Re-run evaluate.py --system all over the full dataset before treating these "
          f"numbers as final.")

print(f"\nWrote {RESULTS_DIR/'metrics_summary.csv'}, "
      f"{RESULTS_DIR/'metrics_summary_variants.csv'}, "
      f"{RESULTS_DIR/'predictions_scored.csv'}, and figures in {FIG_DIR}/")
